<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/from-scratch/cliffwalking-qlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q-Learning para _cliff walking_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---

Ejemplo basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

_Considere el mundo de cuadrícula ("gridworld") mostrado a la derecha. Es una tarea episódica estándar, sin descuento, con estados inicial y meta, y con las acciones habituales que provocan un movimiento hacia arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cliffwalking.png?raw=true" width=500/>

> Udacity (2019) Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/blob/master/temporal-difference/Temporal_Difference_Solution.ipynb).

## Entorno

[Gymnasium](https://gymnasium.farama.org/), antes conocido como Open AI Gym, es un estándar de API para el aprendizaje por refuerzo con una amplia colección de entornos de referencia.


In [ ]:
%pip install "gymnasium[toy-text]>=1.1"


[Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/) se incluye como entorno de ejemplo (y sencillo).

<img src="https://gymnasium.farama.org/_images/cliff_walking.gif"/>

Podemos crear una instancia del entorno, el objeto [`env`](https://gymnasium.farama.org/api/env/#gymnasium-env), e interactuar con él mediante una interfaz común que implementa las operaciones básicas del bucle agente-entorno.

<img src="https://github.com/jgromero/drl-csic/blob/es/img/rl-loop.png?raw=true" width=300/>

Las funciones principales son las siguientes (según la [documentación](https://gymnasium.farama.org/api/env/#) de la API de Gymnasium):

*   [`reset`](https://gymnasium.farama.org/api/env/#gymnasium.Env.reset): Reinicia el entorno a un estado inicial; es necesario llamarla antes de `step`. Devuelve la primera observación del agente para un episodio e información adicional (métricas, información de depuración).
*   [`step`](https://gymnasium.farama.org/api/env/#gymnasium.Env.step): Actualiza el entorno con las acciones y devuelve la siguiente observación del agente, la recompensa por realizar esa acción, si el entorno ha terminado (_terminated_) o se ha truncado (_truncated_) debido a la última acción, e información del entorno sobre el paso (métricas, información de depuración).
*   [`render`](https://gymnasium.farama.org/api/env/#gymnasium.Env.render): Renderiza el entorno para ayudar a visualizar lo que ve el agente; algunos modos son “human”, “rgb_array” y “ansi” (texto).
*   [`close`](https://gymnasium.farama.org/api/env/#gymnasium.Env.close): Cierra el entorno; es importante cuando se usa software externo (p. ej., pygame para el renderizado, o bases de datos).

A continuación vemos un ejemplo sencillo de un agente aleatorio que interactúa 100 veces con el entorno. Cada acción es una muestra aleatoria tomada del vector [`action_space`](https://gymnasium.farama.org/api/vector/#action_space).

In [ ]:
import gymnasium as gym

# Crear el entorno
env = gym.make("CliffWalking-v1", render_mode="ansi")   # cambiar a "human" en ejecución local

# Iniciar el entorno
observation, info = env.reset(seed=42)

# Interactuar 100 veces con el entorno.
# La acción aplicada en cada paso es una acción aleatoria muestreada del espacio de acciones
for i in range(100):
    action = env.action_space.sample()  # aquí se insertaría la política

    #print(f"Acción seleccionada {action} en el paso {i}")

    observation, reward, terminated, truncated, info = env.step(action)

    #print(f"Recompensa obtenida {reward}\n")
    #print(f"Siguiente estado {observation}")

    if terminated or truncated:   # reiniciar el entorno si ha terminado
        observation, info = env.reset()

env.close()

Los detalles de la definición del entorno se encuentran en la [documentación de Gymnasium sobre Cliff Walking](https://gymnasium.farama.org/environments/toy_text/cliff_walking/).

Para ver cómo actúa un agente, la función `show_episode()` ejecuta un episodio completo y muestra las imágenes generadas por el entorno (`render_mode = "rgb_array"`) como una animación, tanto en Google Colab como en local. La usaremos para comparar el comportamiento del agente antes y después del entrenamiento.

In [ ]:
import io
from PIL import Image
from IPython.display import display, Image as IPImage

def show_episode(env, act, max_t=500):
    """Ejecuta un episodio y lo muestra en el cuaderno como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción para un estado
        max_t (int): número máximo de pasos del episodio
    """
    state, _ = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    for t in range(max_t):
        state, reward, terminated, truncated, _ = env.step(act(state))
        frames.append(Image.fromarray(env.render()))
        score += reward
        if terminated or truncated:
            break

    # guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format='gif', save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata['render_fps'], loop=0)
    display(IPImage(data=gif.getvalue(), format='gif'))
    print('Puntuación:', score)

Veamos, por ejemplo, un episodio de un agente que elige acciones al azar.

In [ ]:
env = gym.make("CliffWalking-v1", render_mode="rgb_array")

show_episode(env, lambda state: env.action_space.sample(), max_t=100)

## Algoritmo Q-Learning
A continuación se muestra una implementación básica del algoritmo Q-Learning. Recordemos la regla de actualización de la tabla $Q$:

$Q(S_t, A_t) ← Q(S_t, A_t) + α·[R_t+γmax_{A_{t+1}}Q(S_{t+1}, A_{t+1})-Q(S_t,A_t)]$

In [ ]:
import gymnasium as gym
import sys
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

def generate_action_from_Q_epsilon(discrete_env, Q, epsilon, state):
    """Generador de acciones: obtiene la acción de la política implícita en Q
    Parámetros
    ==========
    discrete_env: entorno; debe tener un espacio de acciones discreto
    Q: tabla Q, definida como un diccionario Q[estado][acción] = retorno esperado
    epsilon: valor de ε-greedy
    state: estado actual"""

    r = np.random.uniform(size=1)[0]
    if r <= epsilon:
        action = discrete_env.action_space.sample()
    else:
        action = np.argmax(Q[state])

    return action

def qlearning(env, num_episodes, generate_action, step_limit=1000, gamma=1.0, alpha=0.01, epsilon=0.2):
    """Algoritmo Q-Learning: aproxima una política óptima refinando Q
    Parámetros
    ==========
    env: entorno; debe tener un espacio de acciones discreto
    num_episodes: número de episodios de entrenamiento
    generate_action: función que genera la siguiente acción
    step_limit: número máximo de pasos por episodio
    gamma: factor de descuento
    alpha: tasa de aprendizaje
    epsilon: valor de ε-greedy"""

    # inicializar la tabla Q a ceros
    Q = defaultdict(lambda: np.zeros(env.action_space.n))  # Q

    scores = []   # puntuaciones de todos los episodios

    for i_episode in range(1, num_episodes+1):

        # seguimiento del progreso
        if i_episode % 100 == 0:
            print("\rEpisodio {}/{}.".format(i_episode, num_episodes), end="")
            sys.stdout.flush()

        # iniciar el episodio
        state, _ = env.reset()
        episode_score = 0
        n_step = 0

        # epsilon variable
        # epsilon = 1 / i_episode

        # bucle del episodio
        while True:
            # elegir la acción (off-policy)
            action = generate_action(env, Q, epsilon, state)

            # aplicar la acción y actualizar la puntuación del episodio
            next_state, reward, terminated, truncated, _ = env.step(action)
            episode_score += reward

            # actualizar Q
            Q[state][action] = Q[state][action] + alpha * (reward + gamma * np.max(Q[next_state]) - Q[state][action])

            # comprobar el fin del episodio
            n_step += 1
            if terminated or truncated or n_step >= step_limit:
                scores.append(episode_score)
                break

            # obtener el siguiente estado
            state = next_state

    # construir la política a partir de la tabla Q
    policy = dict((k,np.argmax(v)) for k, v in Q.items())

    return Q, policy, scores

In [ ]:
def plot_scores(scores, title, solved_score=None, window=100):
    """Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Puntuación')
    plt.plot(np.arange(len(scores)), avg, label='Puntuación media (últimos {})'.format(window))
    if solved_score is not None:
        plt.axhline(solved_score, color='gray', linestyle='--', label='Resuelto')
    plt.ylabel('Puntuación')
    plt.xlabel('Episodio nº')
    plt.title(title)
    plt.legend()
    plt.show()

Ahora podemos ejecutar la función `qlearning` sobre el entorno.

In [ ]:
env = gym.make("CliffWalking-v1", render_mode="rgb_array")

Q, policy, scores = qlearning(env, 25000, generate_action_from_Q_epsilon)

plot_scores(scores, 'Q-Learning')

La política resultante es la siguiente.

In [ ]:
def print_policy(actions):
    """Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(['↑', '→', '↓', '←'])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = 'C'   # celdas del acantilado
    grid[3, 11] = 'T'     # meta
    print('\n'.join(' '.join(row) for row in grid))

In [ ]:
print_policy([np.argmax(Q[state]) for state in range(48)])

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$.

(Se necesita la función `plot_values()` para la visualización.)

In [ ]:
def plot_values(V):
    # redimensionar la función de valor de estado
    V = np.reshape(V, (4,12))
    # representar la función de valor de estado
    fig = plt.figure(figsize=(15,5))
    ax = fig.add_subplot(111)
    im = ax.imshow(V, cmap='cool')
    for (j,i),label in np.ndenumerate(V):
        ax.text(i, j, '{:.1f}'.format(label), ha='center', va='center', fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title('Función de valor de estado')
    plt.show()

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
V = ([np.max(Q[key]) if key in Q else 0 for key in np.arange(48)])
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4,12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## Visualizar el agente entrenado

Ejecutamos el agente durante un episodio con la política _greedy_ de la tabla $Q$. Las imágenes se muestran como una animación, tanto en Google Colab como en local.

In [ ]:
show_episode(env, lambda state: np.argmax(Q[state]), max_t=100)